Se encontraron 1 imágenes en la página 0
Guardada: imagen_1_1.jpeg


In [3]:
import fitz  # PyMuPDF
import io
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

def procesar_pdf_con_ocr(ruta_pdf):
    """
    Extrae imágenes de un archivo PDF y utiliza TrOCR para leer el texto de cada una.
    """
    # --- 1. Carga el modelo OCR una sola vez ---
    # Esto ahorra mucho tiempo al no recargar el modelo por cada imagen.
    print("Cargando el modelo TrOCR... (esto puede tardar un momento la primera vez)")
    try:
        processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
        model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
        print("✅ Modelo cargado correctamente.")
    except Exception as e:
        print(f"❌ Error al cargar el modelo: {e}")
        print("Asegúrate de tener conexión a internet la primera vez que ejecutas el script.")
        return

    # --- 2. Abre el archivo PDF ---
    try:
        pdf_file = fitz.open(ruta_pdf)
        print(f"📄 Procesando el archivo: {ruta_pdf} ({pdf_file.page_count} páginas)")
    except Exception as e:
        print(f"❌ Error al abrir el PDF: {e}")
        return

    # --- 3. Itera sobre las páginas e imágenes ---
    for page_index in range(len(pdf_file)):
        page = pdf_file[page_index]
        image_list = page.get_images(full=True)

        if not image_list:
            print(f"\n-> Página {page_index + 1}: No se encontraron imágenes.")
            continue
        
        print(f"\n-> Página {page_index + 1}: Se encontraron {len(image_list)} imágenes.")

        for image_index, img in enumerate(image_list, start=1):
            xref = img[0]
            base_image = pdf_file.extract_image(xref)
            image_bytes = base_image["image"]

            print(f"  - Procesando imagen {image_index}...")

            try:
                # --- 4. Procesa la imagen directamente en memoria ---
                # Carga los bytes de la imagen sin guardarla en disco
                image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

                # Procesa con TrOCR para obtener el texto
                pixel_values = processor(images=image, return_tensors="pt").pixel_values
                generated_ids = model.generate(pixel_values)
                generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                # Muestra el resultado
                print(f"    ✅ Texto extraído: '{generated_text}'")

            except Exception as e:
                print(f"    ❌ No se pudo procesar la imagen {image_index}: {e}")

    print("\n🎉 Proceso completado.")


# --- EJECUCIÓN DEL SCRIPT ---
if __name__ == "__main__":
    # Cambia "tu_documento.pdf" por la ruta a tu archivo
    ruta_del_archivo_pdf = "ciencia.pdf"
    procesar_pdf_con_ocr(ruta_del_archivo_pdf)

Cargando el modelo TrOCR... (esto puede tardar un momento la primera vez)


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Modelo cargado correctamente.
📄 Procesando el archivo: ciencia.pdf (1 páginas)

-> Página 1: Se encontraron 1 imágenes.
  - Procesando imagen 1...
    ✅ Texto extraído: ':'

🎉 Proceso completado.


In [5]:
import fitz  # PyMuPDF
import io
from PIL import Image
import easyocr
import torch # EasyOCR puede usar torch

def procesar_pdf_con_ocr_avanzado(ruta_pdf):
    """
    Extrae imágenes de un PDF y utiliza EasyOCR para leer el texto,
    ideal para imágenes con múltiples líneas o párrafos.
    """
    # --- 1. Carga el modelo OCR una sola vez ---
    # Especificamos español e inglés. 'en' es importante para términos científicos.
    print("Cargando el modelo EasyOCR... (esto puede tardar la primera vez)")
    try:
        # Si tienes una GPU NVIDIA, se usará automáticamente.
        reader = easyocr.Reader(['es', 'en'], gpu=torch.cuda.is_available())
        print("✅ Modelo cargado correctamente.")
    except Exception as e:
        print(f"❌ Error al cargar el modelo: {e}")
        return

    # --- 2. Abre el archivo PDF ---
    try:
        pdf_file = fitz.open(ruta_pdf)
        print(f"📄 Procesando el archivo: {ruta_pdf} ({pdf_file.page_count} páginas)")
    except Exception as e:
        print(f"❌ Error al abrir el PDF: {e}")
        return

    # --- 3. Itera sobre las páginas e imágenes ---
    for page_index in range(len(pdf_file)):
        page = pdf_file[page_index]
        image_list = page.get_images(full=True)

        if not image_list:
            print(f"\n-> Página {page_index + 1}: No se encontraron imágenes.")
            continue

        print(f"\n-> Página {page_index + 1}: Se encontraron {len(image_list)} imágenes.")

        for image_index, img in enumerate(image_list, start=1):
            xref = img[0]
            base_image = pdf_file.extract_image(xref)
            image_bytes = base_image["image"]

            print(f"  - Procesando imagen {image_index}...")

            try:
                # --- 4. Procesa la imagen con EasyOCR ---
                # EasyOCR devuelve una lista de resultados (bounding_box, texto, confianza)
                resultados = reader.readtext(image_bytes)

                if not resultados:
                    print("    -> No se detectó texto en esta imagen.")
                    continue

                print("    ✅ Texto extraído:")
                # Imprime cada bloque de texto detectado
                for (bbox, texto, prob) in resultados:
                    print(f"      - '{texto}' (Confianza: {prob:.2f})")

            except Exception as e:
                print(f"    ❌ No se pudo procesar la imagen {image_index}: {e}")

    print("\n🎉 Proceso completado.")


# --- EJECUCIÓN DEL SCRIPT ---
if __name__ == "__main__":
    ruta_del_archivo_pdf = "ciencia.pdf"
    procesar_pdf_con_ocr_avanzado(ruta_del_archivo_pdf)

Using CPU. Note: This module is much faster with a GPU.


Cargando el modelo EasyOCR... (esto puede tardar la primera vez)
❌ Error al cargar el modelo: <urlopen error [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond>


In [6]:
import easyocr

In [8]:
reader = easyocr.Reader(['es'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [9]:
result = reader.readtext('images.png')

c:\Users\josealop\repos\CHATBOT_RAG\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [10]:
result

[([[np.int32(85), np.int32(83)],
   [np.int32(115), np.int32(83)],
   [np.int32(115), np.int32(97)],
   [np.int32(85), np.int32(97)]],
  'Ove',
  np.float64(0.3873202573736215)),
 ([[np.int32(75), np.int32(105)],
   [np.int32(115), np.int32(105)],
   [np.int32(115), np.int32(119)],
   [np.int32(75), np.int32(119)]],
  'AMOR',
  np.float64(0.9997727274894714)),
 ([[np.int32(71), np.int32(127)],
   [np.int32(121), np.int32(127)],
   [np.int32(121), np.int32(141)],
   [np.int32(71), np.int32(141)]],
  'AMORE',
  np.float64(0.988114196100651)),
 ([[np.int32(69), np.int32(149)],
   [np.int32(121), np.int32(149)],
   [np.int32(121), np.int32(163)],
   [np.int32(69), np.int32(163)]],
  'AMOUR',
  np.float64(0.8672202658744724)),
 ([[np.int32(71), np.int32(171)],
   [np.int32(121), np.int32(171)],
   [np.int32(121), np.int32(185)],
   [np.int32(71), np.int32(185)]],
  'AMA R E',
  np.float64(0.48430628218128885))]

In [3]:
# Importamos las clases necesarias de la librería pypdf
from pypdf import PdfReader, PdfWriter
import copy

# --- AJUSTA ESTE VALOR ---
# Define el porcentaje de margen a recortar de la parte superior e inferior de cada diapositiva.
# Por ejemplo, 0.06 significa que se recorta un 6%.
# Si quieres más recorte, aumenta el número (ej: 0.08). Si quieres menos, redúcelo (ej: 0.04).
PORCENTAJE_RECORTE = 0.06

def convertir_presentacion_doble_a_sencilla(input_pdf_path, output_pdf_path):
    """
    Lee un PDF con dos diapositivas por página (una arriba y otra abajo) y lo 
    convierte a un nuevo PDF con una diapositiva por página, recortando los márgenes.
    """
    try:
        pdf_writer = PdfWriter()

        with open(input_pdf_path, "rb") as file:
            pdf_reader = PdfReader(file)
            print(f"El archivo '{input_pdf_path}' tiene {len(pdf_reader.pages)} páginas.")

            for page_num, page in enumerate(pdf_reader.pages):
                page_superior = copy.copy(page)
                page_inferior = copy.copy(page)

                ancho_total = page.mediabox.width
                alto_total = page.mediabox.height
                
                mitad_del_alto = alto_total / 2
                
                # Calculamos cuánto vamos a recortar en base al porcentaje definido
                recorte_vertical = mitad_del_alto * PORCENTAJE_RECORTE

                # --- Lógica de corte ajustada ---

                # Definimos el área para la diapositiva SUPERIOR, ya recortada
                page_superior.cropbox.lower_left = (0, mitad_del_alto + recorte_vertical)
                page_superior.cropbox.upper_right = (ancho_total, alto_total - recorte_vertical)

                # Definimos el área para la diapositiva INFERIOR, ya recortada
                page_inferior.cropbox.lower_left = (0, 0 + recorte_vertical)
                page_inferior.cropbox.upper_right = (ancho_total, mitad_del_alto - recorte_vertical)
                
                pdf_writer.add_page(page_superior)
                pdf_writer.add_page(page_inferior)

        with open(output_pdf_path, "wb") as output_file:
            pdf_writer.write(output_file)
        
        print(f"¡Éxito! ✨ Se ha creado el archivo '{output_pdf_path}' con los márgenes recortados.")

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{input_pdf_path}'.")
    except Exception as e:
        print(f"Ha ocurrido un error inesperado: {e}")

# --- Ejecución del script ---
if __name__ == "__main__":
    archivo_entrada = "../documents/Cliente Ligero SCSP.pdf"
    archivo_salida = "Cliente Ligero SCSP - una por pagina.pdf"
    convertir_presentacion_doble_a_sencilla(archivo_entrada, archivo_salida)

El archivo '../documents/Cliente Ligero SCSP.pdf' tiene 26 páginas.
¡Éxito! ✨ Se ha creado el archivo 'Cliente Ligero SCSP - una por pagina.pdf' con los márgenes recortados.
